# LangGraph 단계별 스트리밍 출력

LangGraph의 `stream()` 메서드는 그래프의 각 단계를 스트리밍하는 기능을 제공합니다. 이 튜토리얼에서는 다양한 스트리밍 모드와 옵션을 살펴봅니다.

> 참고 문서: [LangGraph Streaming](https://langchain-ai.github.io/langgraph/concepts/streaming/)

## 튜토리얼 그래프 설정

아래의 LangGraph 예제는 이전 섹션의 Agent 예제와 동일합니다. Google News 검색 도구를 사용하는 챗봇 에이전트를 구축하고, 이를 통해 다양한 스트리밍 모드를 테스트합니다.

추가로 `dummy_data` 상태 필드를 추가하여 여러 상태 키가 있을 때 `output_keys` 옵션이 어떻게 동작하는지 확인합니다.

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv(override=True)

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("LangGraph-V1-Tutorial")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangGraph-V1-Tutorial


In [13]:
from typing import Annotated, List, Dict
from typing_extensions import TypedDict

from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_teddynote.graphs import visualize_graph
from langchain_teddynote.tools import GoogleNews


########## 1. 상태 정의 ##########
# 상태 정의
class State(TypedDict):
    # 메시지 목록: add_messages reducer를 사용하여 메시지 누적
    messages: Annotated[list, add_messages]
    # 더미 데이터: output_keys 테스트용 (reducer 없이 단순 덮어쓰기)
    dummy_data: str


########## 2. 도구 정의 및 바인딩 ##########
# 도구 초기화
# 키워드로 뉴스 검색하는 도구 생성
news_tool = GoogleNews()


@tool
def search_keyword(query: str) -> List[Dict[str, str]]:
    """Look up news by keyword"""
    news_tool = GoogleNews()
    return news_tool.search_by_keyword(query, k=5)


tools = [search_keyword]

# LLM 초기화
# OpenAI 키를 사용하는 경우 gpt-4.1-mini, gpt-4.1 등으로 변경하세요
llm = init_chat_model("claude-sonnet-4-5")

# 도구와 LLM 결합
llm_with_tools = llm.bind_tools(tools)


########## 3. 노드 추가 ##########
# 챗봇 함수 정의
def chatbot(state: State):
    """챗봇 노드 함수
    
    메시지를 받아 LLM에 전달하고 응답을 반환합니다.
    """
    return {
        "messages": [llm_with_tools.invoke(state["messages"])],
        "dummy_data": "[chatbot] 호출, dummy data",  # 테스트를 위하여 더미 데이터를 추가합니다.
    }


# 상태 그래프 생성
graph_builder = StateGraph(State)

# 챗봇 노드 추가
graph_builder.add_node("chatbot", chatbot)


# 도구 노드 생성 및 추가
tool_node = ToolNode(tools=tools)

# 도구 노드 추가
graph_builder.add_node("tools", tool_node)

# 조건부 엣지
graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)

########## 4. 엣지 추가 ##########

# tools > chatbot
graph_builder.add_edge("tools", "chatbot")

# START > chatbot
graph_builder.add_edge(START, "chatbot")

########## 5. 그래프 컴파일 ##########

# 메모리 체크포인터 생성 (interrupt 기능 사용을 위해 필요)
memory = MemorySaver()

# 그래프 빌더 컴파일
graph = graph_builder.compile(checkpointer=memory)

위 코드에서 생성한 그래프의 구조는 다음과 같습니다. `chatbot` 노드에서 `tools_condition`에 따라 조건부 분기가 이루어지며, 도구 호출이 필요한 경우 `tools` 노드로, 그렇지 않으면 `__end__`로 이동합니다.

![chatbot-graph](./assets/chatbot-graph.png)

## StateGraph의 `stream` 메서드

`stream` 메서드는 단일 입력에 대한 그래프 단계를 스트리밍하는 기능을 제공합니다. 각 노드의 실행 결과를 순차적으로 받아볼 수 있어, 실시간으로 진행 상황을 모니터링하거나 점진적으로 결과를 표시할 때 유용합니다.

**매개변수**
- `input` (Union[dict[str, Any], Any]): 그래프에 대한 입력
- `config` (Optional[RunnableConfig]): 실행 구성 (thread_id, recursion_limit 등)
- `stream_mode` (Optional[Union[StreamMode, list[StreamMode]]]): 출력 스트리밍 모드
- `output_keys` (Optional[Union[str, Sequence[str]]]): 스트리밍할 키
- `subgraphs` (bool): 하위 그래프 스트리밍 여부

**반환값**
- Iterator[Union[dict[str, Any], Any]]: 그래프의 각 단계 출력. 출력 형태는 `stream_mode`에 따라 다름

**스트리밍 모드**
- `values`: 각 단계의 현재 상태 값 출력
- `updates`: 각 단계의 상태 업데이트만 출력 (기본값)
- `debug`: 각 단계의 디버그 이벤트 출력

**참고: interrupt 옵션**

`interrupt_before`와 `interrupt_after` 옵션은 **`compile()` 메서드**에서 설정해야 합니다. `stream()` 메서드가 아닌 그래프 컴파일 시점에 지정합니다.

```python
# 올바른 사용법: compile()에서 interrupt 설정
graph = graph_builder.compile(
    checkpointer=memory,
    interrupt_before=["tools"],  # tools 노드 실행 전 중단
)
```

In [4]:
from langchain_core.runnables import RunnableConfig

# 질문
question = "2024년 노벨 문학상 관련 뉴스를 알려주세요."

# 초기 입력 상태를 정의
input = State(dummy_data="테스트 문자열", messages=[("user", question)])

# config 설정
config = RunnableConfig(
    recursion_limit=10,  # 최대 10개의 노드까지 방문. 그 이상은 RecursionError 발생
    configurable={"thread_id": "1"},  # 스레드 ID 설정
    tags=["my-tag"],  # Tag
)

## 그래프 실행 및 스트리밍 테스트

`config` 설정과 함께 스트리밍 출력을 진행합니다. 기본 스트리밍 모드는 `updates`이며, 각 노드의 실행 결과를 순차적으로 출력합니다.

아래 코드에서는 질문을 입력하고 그래프를 스트리밍 방식으로 실행합니다.

In [5]:
for event in graph.stream(input=input, config=config):
    for key, value in event.items():
        print(f"\n[ {key} ]\n")
        # value 에 messages 가 존재하는 경우
        if "messages" in value:
            messages = value["messages"]
            # 가장 최근 메시지 1개만 출력합니다.
            value["messages"][-1].pretty_print()


[ chatbot ]

================================== Ai Message ==================================
Tool Calls:
  search_keyword (call_96Sw1QepxIyZSArSPTaDz8sZ)
 Call ID: call_96Sw1QepxIyZSArSPTaDz8sZ
  Args:
    query: 2024 노벨 문학상

[ tools ]

================================= Tool Message =================================
Name: search_keyword

[{"url": "https://news.google.com/rss/articles/CBMibkFVX3lxTFA4YVpqdTZvWURMMWxtbFFzVVBDZkc3ZmMtOEpEV0hZQ3JoUEtyWmFqN21uZkthdW1EM3lwOW1vUlBnU2R3SkpQWDVaa2FQLTBUY1gwWnlhWVFkMDcwdENrNHpEcVhaTXZLX2JFSlBR?oc=5", "content": "[1년전 오늘] 한강 작가, 2024 노벨문학상 시상식 참석 - 전국매일신문"}, {"url": "https://news.google.com/rss/articles/CBMiUkFVX3lxTE9ZSHIzUEQxcFlXSE1GUDR0MWJPZHdWTVFrcm9qaVQ3MTBtU2tsUVZrTjlxMFJTeWctYlIxUElVRmJNTjl1aE1nQVJLNUVSRk82cUE?oc=5", "content": "속보노벨문학상 한강 \"2024년에 계엄 상황 충격받았다\" - 서울경제"}, {"url": "https://news.google.com/rss/articles/CBMiWkFVX3lxTE0zaHd3RVowVldBdW1COEtkS29Lc2NXVkUxb0d6OW1NVzhtUEJuNVZrenRxX0pYMFV2TGlhQkRaQkpBS2pLYkRMS2dIck4xcktIcE1sVDBZaF

### `output_keys` 옵션

`output_keys` 옵션은 스트리밍할 키를 지정하는 데 사용됩니다.

list 형식으로 지정할 수 있으며, **channels 에 정의된 키 중 하나** 여야 합니다.

**팁(tip)**

- 매 단계마다 출력되는 State key 가 많은 경우, 일부만 스트리밍하고 싶은 경우에 유용합니다.

In [6]:
# channels 에 정의된 키 목록을 출력합니다.
print(list(graph.channels.keys()))

['messages', 'dummy_data', '__start__', '__pregel_tasks', 'branch:to:chatbot', 'branch:to:tools']


In [7]:
# 질문
question = "2024년 노벨 문학상 관련 뉴스를 알려주세요."

# 초기 입력 State 를 정의
input = State(dummy_data="테스트 문자열", messages=[("user", question)])

# config 설정
config = RunnableConfig(
    recursion_limit=10,  # 최대 10개의 노드까지 방문. 그 이상은 RecursionError 발생
    configurable={"thread_id": "1"},  # 스레드 ID 설정
    tags=["my-rag"],  # Tag
)

for event in graph.stream(
    input=input,
    config=config,
    output_keys=["dummy_data"],  # messages 를 추가해 보세요!
):
    for key, value in event.items():
        # key 는 노드 이름
        print(f"\n[ {key} ]\n")

        # dummy_data 가 존재하는 경우
        if value:
            # value 는 노드의 출력값
            print(value.keys())
            # dummy_data key 가 존재하는 경우
            if "dummy_data" in value:
                print(value["dummy_data"])


[ chatbot ]

dict_keys(['dummy_data'])
[chatbot] 호출, dummy data

[ tools ]


[ chatbot ]

dict_keys(['dummy_data'])
[chatbot] 호출, dummy data


In [8]:
# 질문
question = "2024년 노벨 문학상 관련 뉴스를 알려주세요."

# 초기 입력 State 를 정의
input = State(dummy_data="테스트 문자열", messages=[("user", question)])

# config 설정
config = RunnableConfig(
    recursion_limit=10,  # 최대 10개의 노드까지 방문. 그 이상은 RecursionError 발생
    configurable={"thread_id": "1"},  # 스레드 ID 설정
    tags=["my-rag"],  # Tag
)

for event in graph.stream(
    input=input,
    config=config,
    output_keys=["messages"],  # messages 만 출력
):
    for key, value in event.items():
        # messages 가 존재하는 경우
        if value and "messages" in value:
            # key 는 노드 이름
            print(f"\n[ {key} ]\n")
            # messages 의 마지막 요소의 content 를 출력합니다.
            print(value["messages"][-1].content)


[ chatbot ]



[ tools ]

[{"url": "https://news.google.com/rss/articles/CBMiQkFVX3lxTE9QemstQnJSeXZvc2lQWk56dHlFZXEzNGNaMXhmbm5GT2tobHRDcnp2RTROdVlrOUpjdGY2bTloeEJVQQ?oc=5", "content": "부천시립도서관, ‘2025년 베스트 대출 도서’ 발표 - 브레이크뉴스"}, {"url": "https://news.google.com/rss/articles/CBMiXkFVX3lxTE8zbzljanBYZmpabGxRRzU4ZlcxaXlCRWxESWFzQ1NfYmptYXU1cXUydzRlVG41NXA2dkpxWWNDeWhXVkFnZDYwb1p2dmdoRWF2WEdROEttbVZTLVNYQVE?oc=5", "content": "Han Kang’s poetic prose embraces wounds of martial law - 경향신문"}, {"url": "https://news.google.com/rss/articles/CBMiU0FVX3lxTE84bnZBVzYtTGdJWFNnVG5VeFl5b3B1RVJGUkZvZ3dETjFBU3R6Q0VWTXd2M3UwSXltOHR0WVRhSjBkbTljQ3AwYUdjdFdpd3NvSDk4?oc=5", "content": "Han Kang Wins Nobel Prize in Literature: A Moment of Honor - 미주한국일보"}, {"url": "https://news.google.com/rss/articles/CBMiR0FVX3lxTE5xZkFDTzNsZ05QTUNpbUFiQlRZeFZMaDVDTGNKNkt3TXdLeU9qc2xwWld2TXZTZlVDMXpOdUxkWkZUV1FnNlB3?oc=5", "content": "월간 '리더피플', ‘2025년 대한민국 리더 100인’ 선정 - 브레이크뉴스"}, {"url": "https://news.google.com/rss/artic

### `stream_mode` 옵션

`stream_mode` 옵션은 스트리밍 출력 모드를 지정하는 데 사용됩니다.

- `values`: 각 단계의 현재 상태 값 출력 
- `updates`: 각 단계의 상태 업데이트만 출력 (기본값)

아래 다이어그램은 두 모드의 차이를 보여줍니다.

![stream_mode 비교](./assets/stream-mode-comparison.png)

#### `stream_mode = "values"`

`values` 모드는 각 단계의 현재 상태 값을 출력합니다.

**참고**

`event.items()`

- `key`: State 의 key 값
- `value`: State 의 key 에 대한하는 value

In [9]:
# 질문
question = "2024년 노벨 문학상 관련 뉴스를 알려주세요."

# 초기 입력 State 를 정의
input = State(dummy_data="테스트 문자열", messages=[("user", question)])

# config 설정 (새로운 thread_id 사용)
config = RunnableConfig(
    recursion_limit=10,
    configurable={"thread_id": "values-mode-test"},  # thread_id 필수
    tags=["my-rag"],
)

# values 모드로 스트리밍 출력
for event in graph.stream(
    input=input,
    config=config,  # checkpointer 사용 시 config 필수
    stream_mode="values",
):
    for key, value in event.items():
        # key 는 state 의 key 값
        print(f"\n[ {key} ]\n")
        if key == "messages":
            print(f"메시지 개수: {len(value)}")
            # print(value)
    print("===" * 10, " 단계 ", "===" * 10)


[ messages ]

메시지 개수: 1

[ dummy_data ]

==============================  단계  ==============================

[ messages ]

메시지 개수: 2

[ dummy_data ]

==============================  단계  ==============================

[ messages ]

메시지 개수: 3

[ dummy_data ]

==============================  단계  ==============================

[ messages ]

메시지 개수: 4

[ dummy_data ]

==============================  단계  ==============================


#### `stream_mode = "updates"`

`updates` 모드는 각 단계에 대한 업데이트된 State 만 내보냅니다. 

- 출력은 노드 이름을 key 로, 업데이트된 값을 values 으로 하는 `dictionary` 입니다.

**참고**

`event.items()`

- `key`: 노드(Node) 의 이름
- `value`: 해당 노드(Node) 단계에서의 출력 값(dictionary). 즉, 여러 개의 key-value 쌍을 가진 dictionary 입니다.

In [10]:
# 질문
question = "2024년 노벨 문학상 관련 뉴스를 알려주세요."

# 초기 입력 State 를 정의
input = State(dummy_data="테스트 문자열", messages=[("user", question)])

# config 설정 (새로운 thread_id 사용)
config = RunnableConfig(
    recursion_limit=10,
    configurable={"thread_id": "updates-mode-test"},  # thread_id 필수
    tags=["my-rag"],
)

# updates 모드로 스트리밍 출력
for event in graph.stream(
    input=input,
    config=config,  # checkpointer 사용 시 config 필수
    stream_mode="updates",
):
    for key, value in event.items():
        # key 는 노드 이름
        print(f"\n[ {key} ]\n")

        # value 는 노드의 출력값
        print(value.keys())

        # value 에는 state 가 dict 형태로 저장(values 의 key 값)
        if "messages" in value:
            print(f"메시지 개수: {len(value['messages'])}")
            # print(value["messages"])
    print("===" * 10, " 단계 ", "===" * 10)


[ chatbot ]

dict_keys(['messages', 'dummy_data'])
메시지 개수: 1
==============================  단계  ==============================

[ tools ]

dict_keys(['messages'])
메시지 개수: 1
==============================  단계  ==============================

[ chatbot ]

dict_keys(['messages', 'dummy_data'])
메시지 개수: 1
==============================  단계  ==============================


## `interrupt_before`와 `interrupt_after` 옵션

LangGraph v1에서 `interrupt_before`와 `interrupt_after` 옵션은 **`compile()` 메서드**에서 설정합니다. 이 옵션들은 Human-in-the-Loop(HITL) 패턴을 구현할 때 유용하며, 특정 노드 실행 전후에 그래프 실행을 일시 중단합니다.

- `interrupt_before`: 지정된 노드 **실행 전**에 중단
- `interrupt_after`: 지정된 노드 **실행 후**에 중단

**중요**: interrupt 기능을 사용하려면 반드시 `checkpointer`가 필요합니다. 체크포인터가 있어야 중단된 지점의 상태를 저장하고, 이후 `invoke(None, config)`로 재개할 수 있습니다.

아래 다이어그램은 두 옵션의 중단 시점 차이를 보여줍니다.

![interrupt_before vs interrupt_after](./assets/interrupt-flow.png)

아래 코드에서는 `interrupt_before`를 사용하여 `tools` 노드 실행 전에 중단되는 그래프를 컴파일합니다.

In [11]:
# interrupt_before를 사용하는 그래프 컴파일
graph_with_interrupt_before = graph_builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["tools"],  # tools 노드 실행 전에 중단
)

# 질문
question = "2024년 노벨 문학상 관련 뉴스를 알려주세요."

# 초기 입력 State를 정의
input = State(dummy_data="테스트 문자열", messages=[("user", question)])

# config 설정 (새로운 thread_id 사용)
config = RunnableConfig(
    recursion_limit=10,
    configurable={"thread_id": "interrupt-before-test"},
)

print("=== interrupt_before=['tools'] 테스트 ===\n")
print("tools 노드 실행 전에 중단됩니다.\n")

for event in graph_with_interrupt_before.stream(
    input=input,
    config=config,
    stream_mode="updates",
):
    for key, value in event.items():
        # key는 노드 이름
        print(f"\n[{key}]\n")

        # value는 노드의 출력값
        if isinstance(value, dict):
            print(f"출력 키: {value.keys()}")
            if "messages" in value:
                print(f"메시지 개수: {len(value['messages'])}")
                # 마지막 메시지 내용 출력
                last_msg = value["messages"][-1]
                if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                    print(f"도구 호출 요청: {last_msg.tool_calls[0]['name']}")
    print("=" * 40)

# 중단 후 상태 확인
state = graph_with_interrupt_before.get_state(config)
print(f"\n현재 상태 - 다음 실행 노드: {state.next}")

=== interrupt_before=['tools'] 테스트 ===

tools 노드 실행 전에 중단됩니다.


[chatbot]

출력 키: dict_keys(['messages', 'dummy_data'])
메시지 개수: 1
도구 호출 요청: search_keyword

[__interrupt__]


현재 상태 - 다음 실행 노드: ('tools',)


### `interrupt_after` 예제

`interrupt_after`는 지정된 노드가 **실행된 후**에 그래프를 중단합니다. 도구 실행 결과를 확인한 후 다음 단계로 진행할지 결정하는 시나리오에 유용합니다.

아래 코드에서는 `interrupt_after`를 사용하여 `tools` 노드 실행 후에 중단되는 그래프를 컴파일합니다.

In [12]:
# interrupt_after를 사용하는 그래프 컴파일
graph_with_interrupt_after = graph_builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["tools"],  # tools 노드 실행 후에 중단
)

# 질문
question = "2024년 노벨 문학상 관련 뉴스를 알려주세요."

# 초기 입력 State를 정의
input = State(dummy_data="테스트 문자열", messages=[("user", question)])

# config 설정 (새로운 thread_id 사용)
config = RunnableConfig(
    recursion_limit=10,
    configurable={"thread_id": "interrupt-after-test"},
)

print("=== interrupt_after=['tools'] 테스트 ===\n")
print("tools 노드 실행 후에 중단됩니다.\n")

for event in graph_with_interrupt_after.stream(
    input=input,
    config=config,
    stream_mode="updates",
):
    for key, value in event.items():
        # key는 노드 이름
        print(f"\n[{key}]\n")

        # value는 노드의 출력값
        if isinstance(value, dict):
            print(f"출력 키: {value.keys()}")
            if "messages" in value:
                print(f"메시지 개수: {len(value['messages'])}")
    print("=" * 40)

# 중단 후 상태 확인
state = graph_with_interrupt_after.get_state(config)
print(f"\n현재 상태 - 다음 실행 노드: {state.next}")

=== interrupt_after=['tools'] 테스트 ===

tools 노드 실행 후에 중단됩니다.


[chatbot]

출력 키: dict_keys(['messages', 'dummy_data'])
메시지 개수: 1

[tools]

출력 키: dict_keys(['messages'])
메시지 개수: 1

[__interrupt__]


현재 상태 - 다음 실행 노드: ('chatbot',)
